In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import json
import os

print("Loading dataset...")
df = pd.read_csv('datasets/mskcc-skin-tone-labeling-dataset_metadata_2025-11-24.csv')
df = df.dropna(subset=['fitzpatrick_skin_type'])

# Create patient splits
unique_patients = df['patient_id'].unique()
train_patients, temp_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)
val_patients, test_patients = train_test_split(temp_patients, test_size=0.5, random_state=42)

# Filter test set
test_df = df[df['patient_id'].isin(test_patients)].copy()

# Add labels and paths
def group_fitzpatrick(skin_type):
    if skin_type in ['I', 'II']:
        return 'Light'
    elif skin_type in ['III', 'IV']:
        return 'Medium'
    else:
        return 'Dark'

test_df['target_label'] = test_df['fitzpatrick_skin_type'].apply(group_fitzpatrick)
test_df['path'] = test_df['isic_id'].apply(lambda x: f'datasets/MSKCC-images/{x}.jpg')

# Create one-hot labels
label_columns = sorted(test_df['target_label'].unique())
test_df = pd.concat([test_df, pd.get_dummies(test_df['target_label'], dtype='float32')], axis=1)
test_labels = test_df[label_columns].values

print(f"✓ Test set ready: {len(test_df)} images from {len(test_patients)} patients")
print(f"  Classes: {label_columns}")

In [2]:
from tensorflow.keras.models import load_model

# Define focal loss
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        ce = -y_true * tf.math.log(y_pred)
        focal_weight = tf.pow(1 - y_pred, gamma)
        focal_loss_val = alpha * focal_weight * ce
        return tf.reduce_sum(focal_loss_val, axis=-1)
    return loss_fn

# Load model
model = load_model('outputs/best_finetuned_model.keras', 
                   custom_objects={'loss_fn': focal_loss()})

# Build dataset
IMAGE_SIZE = 260
BATCH_SIZE = 8

def parse_image(filepath, label, image_size=IMAGE_SIZE):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [image_size, image_size])
    return image, label

def build_dataset(df, label_columns, image_size, batch_size, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values, df[label_columns].values))
    if is_training:
        ds = ds.shuffle(buffer_size=len(df))
    ds = ds.map(lambda x, y: parse_image(x, y, image_size), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

test_dataset = build_dataset(test_df, label_columns, IMAGE_SIZE, BATCH_SIZE, is_training=False)

print("✓ Model and dataset ready")

✓ Model and dataset ready


d:\skin cancer project\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 621 variables whereas the saved optimizer has 625 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
from sklearn.metrics import classification_report

# Make predictions
print("Making predictions on test set...")
test_predictions = model.predict(test_dataset, verbose=1)

# Evaluate
print("\n=== TEST SET EVALUATION ===")
print(classification_report(
    test_labels.argmax(axis=1),
    test_predictions.argmax(axis=1),
    target_names=label_columns
))

print(f"\nAccuracy: {(test_predictions.argmax(axis=1) == test_labels.argmax(axis=1)).mean():.2%}")

Making predictions on test set...
 9/68 ━━━━━━━━━━━━━━━━━━━━ 1:52 2s/step

In [ ]:
# Verify no train-test overlap
train_test_overlap = set(train_patients) & set(test_patients)
assert len(train_test_overlap) == 0, "ERROR: Train-test overlap!"
print(f"✓ No data leakage: {len(train_patients)} train, {len(test_patients)} test patients")
print(f"✓ Test set: {len(test_df)} images")

In [ ]:
import json

# Load the EXACT patient splits used in training
with open('outputs/patient_splits.json', 'r') as f:
    patient_splits = json.load(f)

train_patients = patient_splits['train']
val_patients = patient_splits['val']
test_patients = patient_splits['test']

# Now filter your dataframe
test_df = df[df['patient_id'].isin(test_patients)].copy()

# VERIFY no overlap
assert len(set(train_patients) & set(test_patients)) == 0, "ERROR: Train-test overlap!"
print(f"✓ Test set verified: {len(test_patients)} patients, {len(test_df)} images")

In [ ]:
# Prepare test dataset
label_columns = sorted(test_df['target_label'].unique())
test_labels = test_df[label_columns].values

# Make predictions
test_dataset = build_dataset(test_df, label_columns, IMAGE_SIZE, BATCH_SIZE, is_training=False)
test_predictions = model.predict(test_dataset)

# Run all evaluations
from sklearn.metrics import classification_report, confusion_matrix

# 1. Classification Report
print("\n=== TEST SET EVALUATION ===")
print(classification_report(
    test_labels.argmax(axis=1),
    test_predictions.argmax(axis=1),
    target_names=label_columns
))

# 2. Confusion Matrix
plot_confusion_matrix(test_labels, test_predictions, label_columns)

# 3. Precision-Recall Curves
plot_precision_recall_curves(test_labels, test_predictions, label_columns)

# 4. Error Analysis with Images
show_classification_errors(test_df, test_predictions, test_labels, label_columns, "Test Set")

# 5. Class-wise Accuracy by Fitzpatrick Type
class_wise_accuracy_by_fitzpatrick(test_df, test_predictions, test_labels, label_columns)

In [ ]:
# Check patient overlap (should be ZERO)
train_test_overlap = set(train_patients) & set(test_patients)
print(f"Train-Test patient overlap: {len(train_test_overlap)}")  # Must be 0!

# Verify test patients
print(f"\nTest patients: {sorted(test_patients)}")
print(f"Test set size: {len(test_df)} images from {len(test_patients)} patients")